# II. Стратегии и тактики

Как солверы устроены внутри? Как они находят решения? Как научиться думать так же -- быстро и формально, как они? :)

В Z3 и подобных ему решателях воплощены тесно связанные друг с другом наборы эвристических алгоритмов доказательства, которые тщательно отбирались и настраивались вручную. Такие наборы, комбинации эвристик, очень хорошо настроены на решение ряда известных классов проблем, однако они начинают плохо работать, когда встречаются новые классы задач. Пока, по сути, солверы могут "научиться" решать новые классы задач, только когда человек-специалист хорошо "объяснит" им подходящие стратегии поиска решений.

С другой стороны, именно потому, что эти эвристики создаются и прорабатываются людьми, они достаточно легко понимаются, и их можно фактически использовать as is, отчуждённо от солверов -- при самостоятельном решении задачек.

---

*Подготовлено Высшей школой программирования Сергея Бобровского 2023*

https://vk.com/lambda_brain

Ссылки на оригинальные ноутбуки в первом занятии.


In [41]:
# !pip install "z3-solver"
from z3 import *

# 1. Тактики и цели

## 1.1. Схема работы Z3

Общая схема работы солвера Z3 такая, что он управляет группой движков логического вывода (это называется **оркестровка**), и на верхнем уровне абстракций "большие" логические шаги решения представляются как функции, называемые **тактики**. Сами тактики объединяются с помощью так называемых **тактических указаний** (tacticals), или **комбинаторов**. Тактики обрабатывают системы ограничений, которые называются **цели** (Goals).

Когда тактика применяется к некоторой цели G, возможны четыре результата.

1) Тактике удаётся показать, что G может быть достигнута (т.е., посильна);

2) Тактике удаётся показать, что G не может быть достигнута (т.е., непосильна);

3) Тактика пока не знает, посильна G или нет, и формирует набор (последовательность) подцелей;

4) Тактика завершается неудачей.

Пункт 3 напоминает очень популярный в программировании подход к проектированию "сверху вниз": декомпозируем сложную задачу на набор более простых, после чего применяем этот подход к каждой подзадаче.

Но когда мы упрощаем (редуцируем, сводим) цель G к последовательности подцелей G1, ..., Gn, мы сталкиваемся с проблемой преобразования нашей модели (системы ограничений) под требования конкретной подцели. Для этого в солверах существует внутренний преобразователь моделей.

## 1.2. Пример цели

In [42]:
x, y = Reals('x y') # две вещественные переменные
g  = Goal() # цель
g.add(x > 0, y > 0, x == y + 1 + 1) # добавим цели набор ограничений

# цель выведется просто как набор ограничений
print(g)


[x > 0, y > 0, x == y + 1 + 1]


Далее добавим две встроенные в Z3 тактики решения: `simplify` и `solve-eqs`.

In [43]:
t1 = Tactic('simplify')
t2 = Tactic('solve-eqs')

**simplify** выполняет простые эквивалентные преобразования выражений (например, x+2+x+2 = 2*x+4).

**solve-eqs** -- это классический метод исключения переменных Гаусса (редукция по столбцам), который вдобавок расширен работой не только с линейной арифметикой, но и, вообще, умеет избавляться практически от любых лишних переменных.

In [44]:
# посмотреть список всех тактик:
print(tactics())

# посмотреть краткое описание всех тактик:
# print(describe_tactics())

# посмотреть краткое описание конкретной тактики:
print(tactic_description('simplify'))
print(tactic_description('solve-eqs'))


['subpaving', 'factor', 'nlsat', 'degree-shift', 'eq2bv', 'purify-arith', 'tseitin-cnf-core', 'tseitin-cnf', 'ctx-solver-simplify', 'unit-subsume-simplify', 'fm', 'bvarray2uf', 'propagate-bv-bounds2', 'propagate-bv-bounds', 'bv-slice', 'dt2bv', 'elim-small-bv', 'blast-term-ite', 'collect-statistics', 'demodulator', 'dom-simplify', 'elim-predicates', 'elim-uncnstr2', 'fold-unfold', 'injectivity', 'occf', 'pb-preprocess', 'propagate-values2', 'special-relations', 'split-clause', 'fpa2bv', 'euf-completion', 'qffplra', 'ff2bv', 'horn-simplify', 'horn', 'qffpbv', 'qffp', 'smtfd', 'pqffd', 'qffd', 'ff-unique', 'ff-simplify', 'ff-solve', 'ff-sat', 'default', 'nra', 'qfidl', 'qfufbv_ackr', 'qfufbv', 'qfauflia', 'lira', 'lia', 'lra', 'aufnira', 'auflira', 'auflia', 'uflra', 'ufnia', 'qfuf', 'qfnra', 'qfnia', 'qflia', 'solver-subsumption', 'sls-smt', 'qfbv-sls', 'nlqsat', 'qe', 'bv-divrem-bounds', 'bv1-blast', 'diff-neq', 'fix-dl-var', 'aig', 'lia2pb', 'pb2bv', 'add-bounds', 'normalize-bounds', 

In [45]:
# посмотрим на промежуточные результаты
# как работают тактики (булевы формулы традиционно выводятся в инвертированном виде):
print(t1(g))

[[Not(x <= 0), Not(y <= 0), x == 2 + y]]


В данном случае упрощено было 2+2 = 4.

In [46]:
print(t2(g))

[[Not(y <= -2), Not(y <= 0)]]


А теперь последнее ограничение `x == y + 1 + 1` тактикой было удалено, потому что оно никак не влияет на другие ограничения.

Далее объединим эти две тактики в комбинатор `Then`. Он работает так, что первый параметр (тактика simplify) применяется к входной цели, а второй параметр (тактика solve-eqs) применяется к каждой подцели, сгенерированной первой тактикой. В данном случае упрощать особо нечего, поэтому и подцель будет одна.

In [47]:
t = Then(t1, t2)
print(t(g))

[[Not(x <= 0), Not(x <= 2)]]


Переменная x полностью исключена из системы ограничений, потому что для любого значения y, которое положительно, всегда найдётся значение x, большее его на 2 -- переменная x никаких дополнительных ограничений не добавляет.


## 1.3. Клаузы

В Z3 называется **клаузой** любое ограничение в форме Or(f_1, ..., f_n). Тактика `split-clause` получает клаузу Or(f_1, ..., f_n) в качестве входной цели, и разбивает её на n подцелей (по одной на каждую субформулу f_i).

In [48]:
x, y, z = Reals('x y z')
g = Goal()
g.add(Or(x == 0, x == 1),
      Or(y == 0, y == 1),
      Or(z == 0, z == 1),
      x + y + z > 2)
t = Tactic('split-clause')
r = t(g) # берём первую клаузу цели и разбиваем её на соответствующие варианты
for g in r:
    print(g)

[x == 0,
 Or(y == 0, y == 1),
 Or(z == 0, z == 1),
 x + y + z > 2]
[x == 1,
 Or(y == 0, y == 1),
 Or(z == 0, z == 1),
 x + y + z > 2]


Мы получили две клаузы -- первая логическая связка Or была разбита на два варианта. Соответственно, можно продолжать применять данную тактику к полученным подцелям.

In [49]:
for g in r:
    print(t(g))

[[x == 0, y == 0, Or(z == 0, z == 1), x + y + z > 2],
 [x == 0, y == 1, Or(z == 0, z == 1), x + y + z > 2]]
[[x == 1, y == 0, Or(z == 0, z == 1), x + y + z > 2],
 [x == 1, y == 1, Or(z == 0, z == 1), x + y + z > 2]]


От всех клауз на этом уровне декомпозиции мы избавились, и далее можно применять другие тактики, подходящие для решения данной системы ограничений.

## 1.4. Комбинаторы тактик

В Z3 доступны следующие комбинаторы тактик (тактические указания):

* `Then(t, s)` применяет тактику t к входной цели, и s к каждой подцели результата t;

* `OrElse(t, s)` применяет тактику t к входной цели, и если она заканчивается неудачей, возвращает результат применения тактики s к входной цели;

* `Repeat(t)` применяет тактику t снова и снова, пока не останется ни одной подцели;

* `Repeat(t, n)` применяет тактику t снова и снова, пока не останется ни одной подцели, но количество попыток не должно превышать заданный порог n;

* `TryFor(t, ms)` применяет тактику t к входной цели, и если она не отрабатывает за ms миллисекунд, возвращается неудача;

* `With(t, params)` применяет тактику t с заданными параметрами params.

In [50]:
# пример
x, y, z = Reals('x y z')
g = Goal()
g.add(Or(x == 0, x == 1),
      Or(y == 0, y == 1),
      Or(z == 0, z == 1),
      x + y + z > 2)

# один шаг разбиения на клаузы:
t = Tactic('split-clause')
print(t(g))


[[x == 0,
  Or(y == 0, y == 1),
  Or(z == 0, z == 1),
  x + y + z > 2],
 [x == 1,
  Or(y == 0, y == 1),
  Or(z == 0, z == 1),
  x + y + z > 2]]


Видно, что была разобрана только одна клауза, а остальные остались неразобранными. Чтобы перевести все клаузы в разобранный вид, воспользуемся комбинатором `Repeat`.

In [51]:
try:
    split_all = Repeat(Tactic('split-clause')) # вылетит с исключением
    print(split_all(g))
except Z3Exception as e:
    print(e.value)

b'split-clause tactic failed, goal does not contain any clause'


Теперь возникнет исключение, потому что, когда все клаузы будут разобраны, `split-clause` завершится неудачей (нету значений, с которыми она могла бы работать).

На такой случай надо предусмотреть стандартный вариант: к итоговому результату, когда основная тактика отработает и завершится неудачей, применить тактику `skip`, которая ничего не делает (просто возвращает входную цель). Для этого воспользуемся комбинатором `OrElse`.

In [52]:
split_all = Repeat(OrElse(Tactic('split-clause'),
                          Tactic('skip')))
print(split_all(g))

for s in split_all(g): # или перебрать подцели по одной
  print(s)

[[x == 0, y == 0, z == 0, x + y + z > 2],
 [x == 0, y == 0, z == 1, x + y + z > 2],
 [x == 0, y == 1, z == 0, x + y + z > 2],
 [x == 0, y == 1, z == 1, x + y + z > 2],
 [x == 1, y == 0, z == 0, x + y + z > 2],
 [x == 1, y == 0, z == 1, x + y + z > 2],
 [x == 1, y == 1, z == 0, x + y + z > 2],
 [x == 1, y == 1, z == 1, x + y + z > 2]]
[x == 0, y == 0, z == 0, x + y + z > 2]
[x == 0, y == 0, z == 1, x + y + z > 2]
[x == 0, y == 1, z == 0, x + y + z > 2]
[x == 0, y == 1, z == 1, x + y + z > 2]
[x == 1, y == 0, z == 0, x + y + z > 2]
[x == 1, y == 0, z == 1, x + y + z > 2]
[x == 1, y == 1, z == 0, x + y + z > 2]
[x == 1, y == 1, z == 1, x + y + z > 2]


In [53]:
# можно ограничиться одной итерацией, тогда не все клаузы будут разобраны:
split_at_most_2 = Repeat(OrElse(Tactic('split-clause'),
                          Tactic('skip')),
                         1) # ограничение на количество проб основной тактики
print(split_at_most_2(g))


[[x == 0, y == 0, Or(z == 0, z == 1), x + y + z > 2],
 [x == 0, y == 1, Or(z == 0, z == 1), x + y + z > 2],
 [x == 1, y == 0, Or(z == 0, z == 1), x + y + z > 2],
 [x == 1, y == 1, Or(z == 0, z == 1), x + y + z > 2]]


Вернёмся к предыдущей Repeat, когда все клаузы были разобраны, и система получилась прозрачной: к ней наверняка подойдёт метод Гаусса. Соответствующую тактику solve-eqs надо применять, когда полностью отработает тактика split-clause, для этого задействуем комбинатор Then.

In [54]:
split_solve = Then(Repeat(OrElse(Tactic('split-clause'),
                                 Tactic('skip'))),
                   Tactic('solve-eqs'))
print(split_solve(g))

[[]]


В результате порождена одна (пустая) трививальная цель, которая достижима по определению.

# 2. Переводим тактики в стратегию

## 2.1. Объединяем тактики в солвер

Набор тактик (**стратегия**) может быть преобразован в конкретный солвер (например, для решения класса похожих задач) с помощью метода `solver()`. Если тактика выдаёт пустую цель, то соответствующий солвер возвращает `sat`. Если тактика выдаёт единственную цель, содержащую False, то солвер возвращает `unsat`. В остальных случаях он возвращает `unknown`.

In [55]:
# создаём солвер на основе комбинаторов и тактик
split_solver = Then(Repeat(OrElse(Tactic('split-clause'),
                                 Tactic('skip'))),
                   Tactic('solve-eqs')).solver()

# добавляем солверу систему ограничений
split_solver.add(Or(x == 0, x == 1))
split_solver.add(Or(y == 0, y == 1))
split_solver.add(Or(z == 0, z == 1))
split_solver.add(x + y + z > 2)

# проверяем, есть ли решение?
print(split_solver.check()) # sat

m = split_solver.model() # вытаскиваем модель

# вычисляем результирующие значения нужных переменных
print(m.evaluate(x))
print(m.evaluate(y))
print(m.evaluate(z))

# или сразу так:
print(m)

sat
1
1
1
[x = 1, z = 1, y = 1]


## 2.2. Стратегия для битовых формул

Подберём подходящую стратегию для решения битовых выражений. В дополнение к тактикам `simplify` и `solve-eqs` дополнительно понадобятся тактики `bit-blast` (алгоритмы bit-blasting для решения сложных и больших битовых формул) и `sat` (классический пропозиционный решатель для работы с формулами, где все переменные логических типов).

Обратите внимание, что для параметров Then явно не указывается Tactic(): все комбинаторы Z3 автоматически вызывают Tactic() для параметра, если он строкового типа.

Команда `solve_using` -- вариант команды `solve`, для которой дополнительно специфицируется конкретный солвер.

In [56]:
bv_solver = Then('simplify',
                 'solve-eqs',
                 'bit-blast',
                 'sat').solver()

x, y = BitVecs('x y', 16)

solve_using(bv_solver, x | y == 13, x > y) # покажите решение!

[y = 0, x = 13]


Рассмотрим этот процесс "изнутри", обращением к Z3 API, без solve_using.

In [57]:
# тактика simplify с помощью комбинатора With
# дополнена параметром mul2concat (не знаю, что он значит :)
bv_solver = Then(With('simplify', mul2concat=True),
                 'solve-eqs',
                 'bit-blast',
                 'aig', # ещё одна тактика, которая сжимает логические формулы
                        # с помощью инвертированных графов
                 'sat').solver()
x, y = BitVecs('x y', 16)

# добавляем ограничение солверу:
bv_solver.add(x*32 + y == 13, x & y < 10, y > -100)

# решение есть?
print(bv_solver.check())

m = bv_solver.model()
print(m) # показать решение

# проверочные вычисления
print(x*32 + y, "==", m.evaluate(x*32 + y))
print(x & y, "==", m.evaluate(x & y))

sat
[x = 1280, y = 24589]
x*32 + y == 13
x & y == 0


## 2.3. Ещё примеры стратегий

Стандартный солвер Z3 доступен в виде отдельной тактики `smt`.

In [58]:
x, y = Ints('x y')
s = Tactic('smt').solver()
s.add(x > y + 1)
print(s.check())
print(s.model())

sat
[y = -2, x = 0]


Пример стратегии в формате солвера, которую можно применять для решения систем арифметических уравнений, когда каждая переменная ограничена и снизу, и сверху.

In [59]:
s = Then(With('simplify', arith_lhs=True, som=True),
         'normalize-bounds', 'lia2pb', 'pb2bv',
         'bit-blast', 'sat').solver()
x, y, z = Ints('x y z')
solve_using(s,
            x > 0, x < 10,
            y > 0, y < 10,
            z > 0, z < 10,
            3*y + 2*x == z)


[x = 3, y = 1, z = 9]


In [60]:
# В случае же, если переменные ограничений не имеют, солвер откажется работать
s.reset() # очищаем солвер от текущей системы ограничений
solve_using(s, 3*y + 2*x == z) # fail

failed to solve


Тактики можно комбинировать с солверами. Например, мы можем взять конкретную тактику, применить её к цели, получить набор подцелей, затем выбрать одну из подцелей, и решить её с помощью солвера.

In [61]:
# набор тактик
t = Then('simplify',
         'normalize-bounds',
         'solve-eqs')

x, y, z = Ints('x y z')
g = Goal()
g.add(x > 10, y == x + 3, z > y) # цель с ограничениями

r = t(g) # r содержит только одну подцель
print(r)

s = Solver()
s.add(r[0]) # ограничение подцели
print(s.check()) # sat

print(s.model()) # модель для подцели
# преобразовываем эту модель в модель для главной цели с помощью convert_model()
print(r[0].convert_model(s.model()))

[[Not(y <= 13), Not(z <= y)]]
sat
[y = 14, z = 15]
[z = 15, y = 14, x = 11]


Резюме, что мастерство использования SMT-солвера подразумевает прежде всего хорошее знание его тактик решения различных классов задач, и умение их комбинировать. Похожий подход существует и в машинном обучении на базе нейронных сетей, где требуется хорошо знать разные модели и подбирать и настраивать их под конкретную задачу.

## 2.4. Метрики формул
Метрики, меры (probes, formula measures) позволяют оценивать количественные и качественные (да/нет) характеристики различных целей.

In [62]:
describe_probes() # список всех метрик Z3

ackr-bound-probe : A probe to give an upper bound of Ackermann congruence lemmas that a formula might generate.
is-qffplra : true if the goal is in QF_FPLRA.
is-qffpbv : true if the goal is in QF_FPBV (floats+bit-vectors).
is-qffp : true if the goal is in QF_FP (floats).
is-quasi-pb : true if the goal is quasi-pb.
is-qfbv-eq : true if the goal is in a fragment of QF_BV which uses only =, extract, concat.
is-pb : true if the goal is a pseudo-boolean problem.
is-unbounded : true if the goal contains integer/real constants that do not have lower/upper bounds.
is-qfufnra : true if the goal is QF_UFNRA (quantifier-free nonlinear real arithmetic with other theories).
is-lira : true if the goal is in LIRA (linear integer and real arithmetic, formula may have quantifiers).
is-lra : true if the goal is in LRA (linear real arithmetic, formula may have quantifiers).
is-lia : true if the goal is in LIA (linear integer arithmetic, formula may have quantifiers).
is-nira : true if the goal is in NIRA

In [63]:
x, y, z = Reals('x y z')
g = Goal()
g.add(x + y + z > 0)

p = Probe('num-consts') # количество переменных в системе ограничений
print("num-consts:", p(g))

num-consts: 3.0


Комбинатор `FailIf(p)` формирует исключение, если его аргумент выдаёт sat.

Комбинатор `If(p, t1, t2)` работает как условный оператор -- выбирает тактику t1 если p истинно (sat), иначе выбирается тактика t2. If -- это сокращённая запись `OrElse(Then(FailIf(Not(p)), t1), t2)`.

Комбинатор `When(p, t)` -- это сокращённая запись `If(p, t, 'skip')`.

In [64]:
x, y, z = Reals('x y z')
g = Goal()
g.add(x**2 - y**2 >= 0)

p = Probe('num-consts') # 2

# выберем подходящую тактику в зависимости от количества переменных
t = If(p > 2, 'simplify', 'factor')

print(t(g))

g = Goal()
g.add(x + x + y + z >= 0, x**2 - y**2 >= 0)
print(t(g))

[[(y + -1*x)*(y + x) <= 0]]
[[2*x + y + z >= 0, x**2 + -1*y**2 >= 0]]


# Пять детективных задач

Задачи по мотивам книг, фильмов и игры. Числа и обстоятельства в задачах 3–5 придуманы.


In [65]:
from z3 import *


## 1. Расёмон

Самурая убил Тадзёмару, Масако либо он покончил с собой. Причина смерти одна,
орудие тоже одно, меч или кинжал.

Тадзёмару говорит, что убил самурая мечом. Масако видела кинжал в груди мужа.
Считаем это показанием об орудии смерти. Самурай через медиума заявляет,
что заколол себя кинжалом. Дровосек повторяет версию разбойника.

Обозначим причины через t, m, s, меч через w, кинжал через d.

$$
(t\lor m\lor s)\land(\neg t\lor\neg m)
\land(\neg t\lor\neg s)\land(\neg m\lor\neg s)
\land(w\lor d)\land(\neg w\lor\neg d).
$$

Проверьте все показания вместе. Затем оставьте только показание Масако
и разделите причины смерти через `split-clause`. Какие версии остаются возможными?


In [66]:
bandit_killed, wife_killed, suicide = Bools("r_bandit r_wife r_suicide")
sword_used, dagger_used = Bools("r_sword r_dagger")
case_goal = Goal()
case_goal.add(
    Or(bandit_killed, wife_killed, suicide),
    Or(Not(bandit_killed), Not(wife_killed)),
    Or(Not(bandit_killed), Not(suicide)),
    Or(Not(wife_killed), Not(suicide)),
    Or(sword_used, dagger_used),
    Or(Not(sword_used), Not(dagger_used)),
)
all_accounts = Solver()
all_accounts.add(case_goal.as_expr(), bandit_killed, sword_used,
                 dagger_used, suicide)
print("All accounts", all_accounts.check())

case_goal.add(dagger_used)
case_branches = Tactic("split-clause")(case_goal)
for branch_name, branch_goal in zip(
    ["bandit", "wife", "suicide"], case_branches
):
    branch_solver = Solver()
    branch_solver.add(branch_goal.as_expr())
    print(branch_name, branch_solver.check())


All accounts unsat
bandit sat
wife sat
suicide sat


Все показания вместе дают `unsat`. Показание о кинжале допускает все три причины смерти.
Каждая подцель после `split-clause` проверяется отдельно.


## 2. Пляшущие человечки

Расшифруйте записку `1 2 / 3 4 5 4 / 6 7 4 / 8 9 6 10 4 11`.
Одинаковые номера обозначают одинаковые буквы, разные номера обозначают разные.
Буквы кодируются числами от 0 до 25. Словарь кандидатов дан в коде.

Каждое слово должно совпасть с одним из кандидатов своего списка D.

$$
C_j=\bigvee_{w\in D_j}\left(\bigwedge_k x_{c_{jk}}=\operatorname{code}(w_k)\right).
$$

Найдите все варианты. Добавьте подсказку ABE для третьего слова, затем IM для первого.
Сколько решений осталось? Используйте `solve-eqs` и восстановите буквы через `convert_model()`.


In [67]:
cipher_words = [[1, 2], [3, 4, 5, 4], [6, 7, 4], [8, 9, 6, 10, 4, 11]]
word_choices = [
    "IM MY WE HE IT IS IN AT AS AM TO NO ON UP BY".split(),
    "HERE ELSE EVEN THEN WHEN THAT HAVE BEEN COME STAY WAIT".split(),
    "ABE ACE ADA EVE IAN BOB TOM SAM BEN LEO".split(),
    "SLANEY FISHER HOLMES WATSON BARKER MORGAN CARTER".split(),
]
letter_values = {number: Int(f"cipher_{number}") for number in range(1, 12)}

def word_clause(cipher_word, plain_word):
    return And([
        letter_values[number] == ord(character) - ord("A")
        for number, character in zip(cipher_word, plain_word)
    ])

def find_messages(extra_clauses):
    cipher_goal = Goal()
    cipher_goal.add(Distinct(list(letter_values.values())))
    for letter_value in letter_values.values():
        cipher_goal.add(letter_value >= 0, letter_value < 26)
    for cipher_word, candidates in zip(cipher_words, word_choices):
        cipher_goal.add(Or([
            word_clause(cipher_word, candidate)
            for candidate in candidates if len(candidate) == len(cipher_word)
        ]))
    cipher_goal.add(*extra_clauses)
    messages_found = []
    while True:
        reduced_goal = Then("simplify", "solve-eqs")(cipher_goal)[0]
        cipher_solver = Solver()
        cipher_solver.add(reduced_goal.as_expr())
        if cipher_solver.check() != sat:
            break
        full_model = reduced_goal.convert_model(cipher_solver.model())
        message_text = " ".join(
            "".join(chr(full_model.eval(letter_values[number]).as_long() + ord("A"))
                    for number in cipher_word)
            for cipher_word in cipher_words
        )
        messages_found.append(message_text)
        cipher_goal.add(Or([
            letter_value != full_model.eval(letter_value)
            for letter_value in letter_values.values()
        ]))
    return sorted(messages_found)

print(find_messages([]))
print(find_messages([word_clause(cipher_words[2], "ABE")]))
print(find_messages([
    word_clause(cipher_words[2], "ABE"), word_clause(cipher_words[0], "IM")
]))


['IM HERE ABE SLANEY', 'IM HERE ACE SLANEY', 'IT HERE ABE SLANEY', 'IT HERE ACE SLANEY', 'TO HERE ABE SLANEY', 'TO HERE ACE SLANEY', 'UP HERE ABE SLANEY', 'UP HERE ACE SLANEY']
['IM HERE ABE SLANEY', 'IT HERE ABE SLANEY', 'TO HERE ABE SLANEY', 'UP HERE ABE SLANEY']
['IM HERE ABE SLANEY']


Без подсказок получается 8 вариантов, с ABE получается 4, с ABE и IM остаётся один.
Словарь допускает и неграмотные фразы вроде IT HERE. Единственность зависит от словаря.


## 3. 12 разгневанных мужчин

Свидетель говорит, что после шума дошёл до двери за 15 секунд и увидел бегущего человека.
Реакция занимает от 2 до 4 секунд. До двери можно пройти 6 или 12 метров.
На каждый метр нужно от 2 до 3 секунд. Остановок нет.

Обозначим реакцию через r, секунды на метр через p, длину пути через L, полное время через T.

$$
T=r+pL,\quad 2\le r\le4,\quad2\le p\le3,\quad
(L=6\lor L=12),\quad T\le15.
$$

Разделите маршруты через `split-clause` и проверьте, по какому свидетель мог успеть.


In [68]:
reaction_time, seconds_per_meter, route_length, arrival_time = Reals(
    "jury_reaction jury_pace jury_length jury_arrival"
)
witness_goal = Goal()
witness_goal.add(
    Or(route_length == 6, route_length == 12),
    reaction_time >= 2, reaction_time <= 4,
    seconds_per_meter >= 2, seconds_per_meter <= 3,
    arrival_time == reaction_time + seconds_per_meter * route_length,
    arrival_time <= 15,
)
for branch_goal in Tactic("split-clause")(witness_goal):
    route_solver = Then("simplify", "solve-eqs", "smt").solver()
    route_solver.add(branch_goal.as_expr())
    branch_result = route_solver.check()
    print(branch_goal[0], branch_result)
    if branch_result == sat:
        print(route_solver.model())


jury_length == 6 sat
[jury_pace = 2,
 jury_arrival = 14,
 jury_length = 6,
 jury_reaction = 2]
jury_length == 12 unsat


Короткий путь занимает минимум 14 секунд, длинный минимум 26.
Показание совместимо только с коротким маршрутом.


## 4. Silent Hill 2

Джеймсу нужно включить лифт в больнице. Главный и резервный генераторы нельзя включать
одновременно. Для главного нужен исправный предохранитель, для резервного работающий насос.
Насос требует открытого клапана. Клапан должен оставаться закрытым, иначе затопит архив с запиской.

Обозначим главный генератор через g, резервный через b, предохранитель через f,
насос через p, клапан через v.

$$
((g\land f)\lor(b\land p))\land(\neg g\lor\neg b)
\land(\neg p\lor v)\land\neg v.
$$

Переведите формулу в КНФ через `tseitin-cnf` и решите тактикой `sat`.
Проверьте, можно ли включить лифт со сломанным предохранителем.


In [69]:
main_power, backup_power, fuse_ready, pump_active, valve_open = Bools(
    "hill_main hill_backup hill_fuse hill_pump hill_valve"
)
escape_formula = And(
    Or(And(main_power, fuse_ready), And(backup_power, pump_active)),
    Or(Not(main_power), Not(backup_power)),
    Or(Not(pump_active), valve_open),
    Not(valve_open),
)
escape_goal = Goal()
escape_goal.add(escape_formula)
print("CNF", Tactic("tseitin-cnf")(escape_goal))
escape_solver = Then("simplify", "tseitin-cnf", "sat").solver()
escape_solver.add(escape_formula)
print(escape_solver.check())
print(escape_solver.model())
escape_solver.push()
escape_solver.add(Not(fuse_ready))
print("Broken fuse", escape_solver.check())
escape_solver.pop()


CNF [[Or(hill_main, hill_backup),
  Or(hill_fuse, hill_backup),
  Or(hill_main, hill_pump),
  Or(hill_fuse, hill_pump),
  Or(Not(hill_backup), Not(hill_main)),
  Or(Not(hill_pump), hill_valve),
  Not(hill_valve)]]
sat
[hill_main = True,
 hill_backup = False,
 hill_pump = False,
 hill_valve = False,
 hill_fuse = True]
Broken fuse unsat


Закрытый клапан исключает насос и резервный генератор.
Нужны главный генератор и исправный предохранитель.


## 5. Леди Макбет Мценского уезда

Могла ли Катерина сходить из спальни в кладовую до возвращения Сергея?
Через двор идти 4 минуты в каждую сторону, через галерею 7 минут.
В кладовой нужно провести хотя бы 3 минуты. Сергей вернётся на 18-й минуте после полуночи.
Катерине нужно вернуться раньше. Горничная видела её в спальне до 6-й минуты.
Все времена считаем целыми.

Обозначим уход через d, дорогу в одну сторону через u, приход в кладовую через a,
время внутри через h, возвращение через r.

$$
a=d+u,\quad r=a+h+u,\quad d\ge6,\quad h\ge3,\quad r<18,
\quad(u=4\lor u=7).
$$

Примените `solve-eqs`, найдите расписание и восстановите исходные времена через `convert_model()`.
Проверьте отдельно путь через галерею и новое показание горничной об уходе не раньше 8-й минуты.


In [70]:
depart_time, travel_time, arrive_time, inside_time, return_time = Ints(
    "lady_depart lady_travel lady_arrive lady_inside lady_return"
)
alibi_goal = Goal()
alibi_goal.add(
    Or(travel_time == 4, travel_time == 7),
    depart_time >= 6, inside_time >= 3,
    arrive_time == depart_time + travel_time,
    return_time == arrive_time + inside_time + travel_time,
    return_time < 18,
)
reduced_goal = Then("simplify", "solve-eqs")(alibi_goal)[0]
alibi_solver = Solver()
alibi_solver.add(reduced_goal.as_expr())
print(alibi_solver.check())
print(reduced_goal.convert_model(alibi_solver.model()))

for extra_name, extra_clause in [
    ("Gallery", travel_time == 7),
    ("Departure after minute 8", depart_time >= 8),
]:
    check_solver = Then("simplify", "solve-eqs", "smt").solver()
    check_solver.add(alibi_goal.as_expr(), extra_clause)
    print(extra_name, check_solver.check())


sat
[lady_return = 17,
 lady_inside = 3,
 lady_travel = 4,
 lady_arrive = 10,
 lady_depart = 6]
Gallery unsat
Departure after minute 8 unsat


Через двор Катерина может вернуться на 17-й минуте, через галерею только с 23-й.
При уходе с 8-й минуты даже короткий путь позволяет вернуться лишь с 19-й.
